In [1]:
import sqlite3

DB_PATH = 'lab3_blockchain.db'

def get_conn():
    return sqlite3.connect(DB_PATH)

In [7]:
class Block:
    def __init__(self, id, view, desc, img):
        self.id = id
        self.view = view
        self.desc = desc
        self.img = img

    @staticmethod
    def get_all():
        conn = get_conn()
        cur = conn.cursor()
        cur.execute("SELECT id, view, desc, img FROM BLOCKS")
        rows = cur.fetchall()
        conn.close()
        return [Block(row[0], row[1], row[2], row[3]) for row in rows]
        

    @staticmethod
    def get_by_id(block_id):
        conn = get_conn()
        cur = conn.cursor()
        cur.execute("SELECT id, view, desc, img FROM BLOCKS WHERE id = ?", (block_id,))
        row = cur.fetchone()
        conn.close()
        if row:
            return Block(row[0], row[1], row[2], row[3])
        return None

    def __str__(self):
        return f"Block({self.id}, {self.view}, {self.desc})"

In [ ]:
def get_all_blocks():
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("SELECT id, view, desc, img FROM BLOCKS")
    rows = cur.fetchall()
    conn.close()
    blocks = []
    for r in rows:
        blocks.append(Block(r[0], r[1], r[2], r[3]))
    return blocks

In [8]:
class Source:
    def __init__(self, id, ip_addr, country_code):
        self.id = id
        self.ip_addr = ip_addr
        self.country_code = country_code

    @staticmethod
    def get_all():
        conn = get_conn()
        cur = conn.cursor()
        cur.execute("SELECT id, ip_addr, country_code FROM SOURCES")
        rows = cur.fetchall()
        conn.close()
        return [Source(row[0], row[1], row[2]) for row in rows]

    @staticmethod
    def get_by_id(source_id):
        conn = get_conn()
        cur = conn.cursor()
        cur.execute("SELECT id, ip_addr, country_code FROM SOURCES WHERE id = ?", (source_id,))
        row = cur.fetchone()
        conn.close()
        if row:
            return Source(row[0], row[1], row[2])
        return None

    def __str__(self):
        return f"Source({self.id}, {self.ip_addr}, {self.country_code})"

In [9]:
class Person:
    def __init__(self, id, name, addr):
        self.id = id
        self.name = name
        self.addr = addr

    @staticmethod
    def get_all():
        conn = get_conn()
        cur = conn.cursor()
        cur.execute("SELECT id, name, addr FROM PERSONS")
        rows = cur.fetchall()
        conn.close()
        return [Person(row[0], row[1], row[2]) for row in rows]

    @staticmethod
    def get_by_id(person_id):
        conn = get_conn()
        cur = conn.cursor()
        cur.execute("SELECT id, name, addr FROM PERSONS WHERE id = ?", (person_id,))
        row = cur.fetchone()
        conn.close()
        if row:
            return Person(row[0], row[1], row[2])
        return None

    def __str__(self):
        return f"Person({self.id}, {self.name})"

In [10]:
class Vote:
    def __init__(self, block_id, voter_id, timestamp, source_id):
        self.block_id = block_id
        self.voter_id = voter_id
        self.timestamp = timestamp
        self.source_id = source_id

    @staticmethod
    def get_all():
        conn = get_conn()
        cur = conn.cursor()
        cur.execute("SELECT block_id, voter_id, timestamp, source_id FROM VOTES")
        rows = cur.fetchall()
        conn.close()
        return [Vote(row[0], row[1], row[2], row[3]) for row in rows]

    def get_details(self):
        conn = get_conn()
        cur = conn.cursor()
        cur.execute("""
            SELECT 
                v.block_id, v.voter_id, v.timestamp, v.source_id,
                b.view, b.desc,
                p.name, p.addr,
                s.ip_addr, s.country_code
            FROM VOTES v
            LEFT JOIN BLOCKS b ON v.block_id = b.id
            LEFT JOIN PERSONS p ON v.voter_id = p.id
            LEFT JOIN SOURCES s ON v.source_id = s.id
            WHERE v.block_id = ? AND v.voter_id = ?
        """, (self.block_id, self.voter_id))
        row = cur.fetchone()
        conn.close()
        if row:
            return {
                'block_id': row[0],
                'voter_id': row[1],
                'timestamp': row[2],
                'source_id': row[3],
                'block_view': row[4],
                'block_desc': row[5],
                'voter_name': row[6],
                'voter_addr': row[7],
                'source_ip': row[8],
                'source_country': row[9]
            }
        return {}

    def __str__(self):
        return f"Vote({self.block_id}, {self.voter_id}, {self.timestamp})"

In [12]:
print("=== Усі блоки ===")
for b in Block.get_all():
    print("  ", b)

print("\n=== Усі особи ===")
for p in Person.get_all():
    print("  ", p)

print("\n=== Усі джерела ===")
for s in Source.get_all():
    print("  ", s)

print("\n=== Усі голоси ===")
for v in Vote.get_all():
    print("  ", v)

print("\n=== Деталі першого голосу ===")
votes = Vote.get_all()
if votes:
    details = votes[0].get_details()
    for key, value in details.items():
        print(f"  {key}: {value}")

=== Усі блоки ===
   Block(0x1f3a5b7c, 1, First block)
   Block(0x8e2d4f6a, 2, Second block)
   Block(0x9c0b3d1e, 3, Third block)

=== Усі особи ===
   Person(1, Катерина Слободяник)
   Person(2, Юлія Дмишко)
   Person(3, Світлана Удичка)

=== Усі джерела ===
   Source(1, 203.0.113.1, UA)
   Source(2, 198.51.100.2, PL)
   Source(3, 192.0.2.3, DE)
   Source(4, 10.0.0.4, US)

=== Усі голоси ===
   Vote(0x1f3a5b7c, 1, 2025-02-10 12:00:00)
   Vote(0x1f3a5b7c, 2, 2025-02-10 12:05:00)
   Vote(0x8e2d4f6a, 1, 2025-02-10 12:10:00)
   Vote(0x8e2d4f6a, 3, 2025-02-10 12:15:00)
   Vote(0x9c0b3d1e, 2, 2025-02-10 12:20:00)
   Vote(0x9c0b3d1e, 3, 2025-02-10 12:25:00)

=== Деталі першого голосу ===
  block_id: 0x1f3a5b7c
  voter_id: 1
  timestamp: 2025-02-10 12:00:00
  source_id: 1
  block_view: 1
  block_desc: First block
  voter_name: Катерина Слободяник
  voter_addr: Львів, вул. Широка, 63
  source_ip: 203.0.113.1
  source_country: UA
